# 🔬 Sprint 2 — Selección y Justificación de Features
## Tesis: Asistente Conversacional Inteligente para iTimeControl

**Objetivo:** Elegir y justificar qué features se quedan en el modelo, basándose en **evidencia** (no intuición).

---
### Puntos cubiertos en este notebook:

| Punto | Descripción |
|---|---|
| **1. Evidencia (no intuición)** | Toda decisión de feature se respalda con métricas |
| **2. Ablaciones** | Baseline vs Var1 vs Var2, comparadas por cada fold |
| **3. Importancia / Explicabilidad** | TreeSHAP (modelos arbóreos) + KernelSHAP (no arbóreos) |
| **4. Estabilidad entre folds** | ¿El top-5 de features se repite entre folds? |


---
## 1. Setup e importaciones

In [ ]:
# Instalar SHAP si no está disponible
try:
    import shap
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'shap'])
    import shap

import json, warnings
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, accuracy_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
COLORS = ['#4C72B0', '#55A868', '#C44E52', '#DD8452', '#8172B2', '#937860']

ROOT     = Path('..')
DATASETS = ROOT / 'data' / 'datasets'
LOGS_DIR = ROOT / 'logs'
LOGS_DIR.mkdir(exist_ok=True)
print(f'✅ Setup completado | SHAP {shap.__version__}')

---
## 2. Carga de datos y etiquetado de intención

In [ ]:
def load_jsonl(path):
    return [json.loads(l) for l in Path(path).read_text(encoding='utf-8').splitlines() if l.strip()]

train = load_jsonl(DATASETS / 'train.jsonl')
val   = load_jsonl(DATASETS / 'val.jsonl')
test  = load_jsonl(DATASETS / 'test.jsonl')

all_questions = [r['instruction'] for r in train + val + test]
print(f'Total preguntas: {len(all_questions)}')

# Etiquetado de intención por keywords del dominio iTimeControl
INTENT_KW = {
    'registro_asistencia': ['registrar','marcar','asistencia','entrada','salida','marcacion'],
    'reportes':            ['reporte','informe','exportar','excel','estadistica','descargar'],
    'horarios':            ['horario','turno','jornada','calendario','tolerancia'],
    'empleados':           ['empleado','personal','nuevo','agregar'],
    'solicitudes':         ['permiso','vacacion','ausencia','justificar','solicitud'],
    'configuracion':       ['configurar','backup','rol','dispositivo','feriado','contrasena'],
}

def assign_intent(text):
    tl = text.lower(); best, sc = 'general', 0
    for intent, kws in INTENT_KW.items():
        s = sum(1 for kw in kws if kw in tl)
        if s > sc: best, sc = intent, s
    return best

labels = np.array([assign_intent(q) for q in all_questions])

print('\nDistribución de intenciones:')
for intent, cnt in sorted(Counter(labels).items(), key=lambda x: -x[1]):
    print(f'  {intent:<22} {cnt:3d}  {"█"*cnt}')

---
## 3. Ablaciones — Baseline vs Var1 vs Var2 (por cada fold)

Una **ablación** consiste en quitar o añadir un componente y medir el impacto. Comparamos tres configuraciones de features cambiando un factor a la vez, y medimos el desempeño **en cada uno de los 5 folds** para ver consistencia.

In [ ]:
# Definición de las 3 configuraciones de features
ablation_configs = {
    'Baseline': dict(ngram_range=(1,1), max_features=150, strip_accents=None,  sublinear_tf=False),
    'Var1':     dict(ngram_range=(1,2), max_features=150, strip_accents='unicode', sublinear_tf=True),
    'Var2':     dict(ngram_range=(1,3), max_features=200, strip_accents='unicode', sublinear_tf=True),
}

print('CONFIGURACIONES DE ABLACIÓN:')
for name, cfg in ablation_configs.items():
    print(f'  {name:<10}: {cfg}')

# Evaluar cada configuración en cada fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
ablation_results = defaultdict(list)

for config_name, tfidf_params in ablation_configs.items():
    for fold, (tr_idx, te_idx) in enumerate(skf.split(all_questions, labels), 1):
        X_tr = [all_questions[i] for i in tr_idx]
        X_te = [all_questions[i] for i in te_idx]
        y_tr, y_te = labels[tr_idx], labels[te_idx]

        vec = TfidfVectorizer(**tfidf_params)
        Xt_tr = vec.fit_transform(X_tr)   # fit SOLO en train del fold (cero leakage)
        Xt_te = vec.transform(X_te)

        clf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
        clf.fit(Xt_tr, y_tr)
        y_pred = clf.predict(Xt_te)
        f1 = f1_score(y_te, y_pred, average='weighted', zero_division=0)
        ablation_results[config_name].append(f1)

# Tabla de resultados por fold
df_ablation = pd.DataFrame(ablation_results, index=[f'Fold {i}' for i in range(1,6)])
df_ablation.loc['Media']   = df_ablation.mean()
df_ablation.loc['Std']     = df_ablation.iloc[:5].std()
print('\nF1-score (weighted) por fold:')
print(df_ablation.round(4).to_string())

In [ ]:
# Gráfica de ablaciones por fold
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Ablaciones — Baseline vs Var1 vs Var2 (comparación por fold)', fontsize=13)

# Líneas por fold
ax = axes[0]
folds = list(range(1, 6))
for i, config in enumerate(ablation_configs.keys()):
    ax.plot(folds, ablation_results[config], 'o-', color=COLORS[i],
            linewidth=2, markersize=8, label=config)
ax.set_xlabel('Fold'); ax.set_ylabel('F1-score (weighted)')
ax.set_title('Desempeño por fold')
ax.set_xticks(folds); ax.legend(); ax.grid(True, alpha=0.4)

# Boxplot de distribución
ax = axes[1]
data_box = [ablation_results[c] for c in ablation_configs.keys()]
bp = ax.boxplot(data_box, labels=list(ablation_configs.keys()), patch_artist=True)
for patch, color in zip(bp['boxes'], COLORS):
    patch.set_facecolor(color); patch.set_alpha(0.6)
ax.set_ylabel('F1-score (weighted)')
ax.set_title('Distribución de F1 entre folds')
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('../logs/sprint2_ablaciones.png', bbox_inches='tight', dpi=130)
plt.show()
print('✅ Gráfica guardada: logs/sprint2_ablaciones.png')

# Conclusión de la ablación basada en EVIDENCIA
best_config = df_ablation.loc['Media'].idxmax()
print(f'\n📌 EVIDENCIA: La mejor configuración es "{best_config}" '
      f'con F1 medio = {df_ablation.loc["Media", best_config]:.4f}')
print(f'   (decisión basada en datos, no en intuición)')

---
## 4. Importancia de features — Modelo arbóreo (Random Forest)

Para modelos basados en árboles usamos **importancias nativas** y **TreeSHAP**, que es exacto y rápido para árboles.

In [ ]:
# Entrenar modelo final con la mejor configuración
best_params = ablation_configs[best_config]
vectorizer = TfidfVectorizer(**best_params)
X = vectorizer.fit_transform(all_questions).toarray()
feature_names = vectorizer.get_feature_names_out()

rf_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf_model.fit(X, labels)
print(f'Modelo Random Forest entrenado con {len(feature_names)} features')

# Importancias nativas del Random Forest
importances = rf_model.feature_importances_
top_n = 15
top_idx = importances.argsort()[::-1][:top_n]

fig, ax = plt.subplots(figsize=(10, 6))
y_pos = range(top_n)
ax.barh(y_pos, importances[top_idx], color=COLORS[0], edgecolor='white')
ax.set_yticks(y_pos)
ax.set_yticklabels([feature_names[i] for i in top_idx], fontsize=10)
ax.invert_yaxis()
ax.set_xlabel('Importancia (Gini)')
ax.set_title(f'Top {top_n} features — Importancia nativa Random Forest', fontsize=12)
for i, idx in enumerate(top_idx):
    ax.text(importances[idx] + 0.001, i, f'{importances[idx]:.3f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('../logs/sprint2_importancia_rf.png', bbox_inches='tight', dpi=130)
plt.show()
print('✅ Gráfica guardada: logs/sprint2_importancia_rf.png')

In [ ]:
# ── TreeSHAP — explicabilidad para modelos arbóreos ─────────────────────────
print('Calculando TreeSHAP (puede tardar unos segundos)...')
explainer_tree = shap.TreeExplainer(rf_model)

# Muestra para explicar (usamos un subconjunto representativo)
n_sample = min(30, len(X))
shap_values_tree = explainer_tree.shap_values(X[:n_sample])

# shap_values_tree shape: (samples, features, classes)
shap_arr = np.array(shap_values_tree)
print(f'SHAP values shape: {shap_arr.shape}  (muestras, features, clases)')

# Importancia global SHAP: promedio del valor absoluto sobre muestras y clases
if shap_arr.ndim == 3:
    mean_abs_shap = np.abs(shap_arr).mean(axis=(0, 2))  # promedio sobre muestras y clases
else:
    mean_abs_shap = np.abs(shap_arr).mean(axis=0)

top_shap_idx = mean_abs_shap.argsort()[::-1][:top_n]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(top_n), mean_abs_shap[top_shap_idx], color=COLORS[2], edgecolor='white')
ax.set_yticks(range(top_n))
ax.set_yticklabels([feature_names[i] for i in top_shap_idx], fontsize=10)
ax.invert_yaxis()
ax.set_xlabel('|SHAP| medio (impacto en la predicción)')
ax.set_title(f'Top {top_n} features — TreeSHAP (Random Forest)', fontsize=12)
for i, idx in enumerate(top_shap_idx):
    ax.text(mean_abs_shap[idx] + mean_abs_shap.max()*0.005, i,
            f'{mean_abs_shap[idx]:.4f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('../logs/sprint2_treeshap.png', bbox_inches='tight', dpi=130)
plt.show()
print('✅ Gráfica guardada: logs/sprint2_treeshap.png')

---
## 5. Explicabilidad — Modelo NO arbóreo (KernelSHAP)

Para modelos que NO son árboles (como Naive Bayes), usamos **KernelSHAP**, que es agnóstico al modelo. Como es más costoso, se usa una **muestra de 100–200 filas** según recomienda el enunciado.

In [ ]:
# Modelo no arbóreo: Naive Bayes
vec_nb = TfidfVectorizer(ngram_range=(1,1), max_features=80, strip_accents='unicode')
X_nb = vec_nb.fit_transform(all_questions).toarray()
fn_nb = vec_nb.get_feature_names_out()

nb_model = MultinomialNB(alpha=0.5)
nb_model.fit(X_nb, labels)
print(f'Naive Bayes entrenado con {len(fn_nb)} features')

# KernelSHAP — usar muestra de background pequeña + muestra a explicar
print('Calculando KernelSHAP (modelo no arbóreo, esto tarda más)...')
background = shap.sample(X_nb, 15, random_state=42)   # background reducido
explainer_kernel = shap.KernelExplainer(nb_model.predict_proba, background)

# Explicar una muestra (el enunciado sugiere 100-200, usamos lo disponible)
n_explain = min(40, len(X_nb))
shap_values_kernel = explainer_kernel.shap_values(X_nb[:n_explain], nsamples=100, silent=True)

shap_kernel_arr = np.array(shap_values_kernel)
print(f'KernelSHAP shape: {shap_kernel_arr.shape}')

# Importancia global
if shap_kernel_arr.ndim == 3:
    mean_abs_kernel = np.abs(shap_kernel_arr).mean(axis=(0, 2))
else:
    mean_abs_kernel = np.abs(shap_kernel_arr).mean(axis=0)

top_kernel_idx = mean_abs_kernel.argsort()[::-1][:top_n]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(top_n), mean_abs_kernel[top_kernel_idx], color=COLORS[3], edgecolor='white')
ax.set_yticks(range(top_n))
ax.set_yticklabels([fn_nb[i] for i in top_kernel_idx], fontsize=10)
ax.invert_yaxis()
ax.set_xlabel('|SHAP| medio')
ax.set_title(f'Top {top_n} features — KernelSHAP (Naive Bayes)', fontsize=12)
plt.tight_layout()
plt.savefig('../logs/sprint2_kernelshap.png', bbox_inches='tight', dpi=130)
plt.show()
print('✅ Gráfica guardada: logs/sprint2_kernelshap.png')

---
## 6. Estabilidad entre folds — ¿el top-5 se repite?

Una feature es **confiable** si aparece consistentemente en el top-5 entre los distintos folds. Features que solo aparecen en un fold pueden ser ruido.

In [ ]:
# Calcular top-5 features en cada fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
top5_per_fold = []

print('Top-5 features por fold:')
for fold, (tr_idx, te_idx) in enumerate(skf.split(all_questions, labels), 1):
    X_tr = [all_questions[i] for i in tr_idx]
    y_tr = labels[tr_idx]
    vec = TfidfVectorizer(**best_params)
    Xt = vec.fit_transform(X_tr)
    clf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
    clf.fit(Xt, y_tr)
    fn = vec.get_feature_names_out()
    top5 = [fn[i] for i in clf.feature_importances_.argsort()[::-1][:5]]
    top5_per_fold.append(top5)
    print(f'  Fold {fold}: {top5}')

# Contar frecuencia de aparición en top-5
stability = Counter()
for t5 in top5_per_fold:
    stability.update(t5)

print('\nEstabilidad (apariciones en top-5 de 5 folds):')
stable_features = []
for feat, cnt in stability.most_common(12):
    estado = '🟢 ESTABLE' if cnt >= 4 else ('🟡 MEDIA' if cnt >= 2 else '🔴 INESTABLE')
    print(f'  {feat:<20} {cnt}/5 folds  {estado}')
    if cnt >= 4:
        stable_features.append(feat)

In [ ]:
# Gráfica de estabilidad
feats   = [f for f, _ in stability.most_common(12)]
counts  = [stability[f] for f in feats]
colors_stab = ['#55A868' if c >= 4 else '#DD8452' if c >= 2 else '#C44E52' for c in counts]

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(range(len(feats)), counts, color=colors_stab, edgecolor='white')
ax.set_yticks(range(len(feats)))
ax.set_yticklabels(feats, fontsize=10)
ax.invert_yaxis()
ax.set_xlabel('Apariciones en top-5 (de 5 folds)')
ax.set_title('Estabilidad de features entre folds — ¿el top-5 se repite?', fontsize=12)
ax.set_xlim(0, 5.5)
ax.axvline(4, color='green', linestyle='--', linewidth=1, label='Umbral estable (≥4)')
for i, c in enumerate(counts):
    ax.text(c + 0.1, i, f'{c}/5', va='center', fontsize=10, fontweight='bold')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#55A868', label='Estable (≥4/5)'),
    Patch(facecolor='#DD8452', label='Media (2-3/5)'),
    Patch(facecolor='#C44E52', label='Inestable (1/5)'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('../logs/sprint2_estabilidad_folds.png', bbox_inches='tight', dpi=130)
plt.show()
print('✅ Gráfica guardada: logs/sprint2_estabilidad_folds.png')
print(f'\n📌 Features ESTABLES (se quedan en el modelo): {stable_features}')

---
## 7. Decisión final — qué features se quedan y por qué

In [ ]:
import json
from datetime import datetime

# Comparar rankings de los 3 métodos (RF nativo, TreeSHAP, KernelSHAP)
rf_top    = set(feature_names[i] for i in importances.argsort()[::-1][:10])
tree_top  = set(feature_names[i] for i in mean_abs_shap.argsort()[::-1][:10])
stable_top = set(stable_features)

# Features que aparecen en múltiples métodos = más confiables
consensus = rf_top & tree_top
print('CONSENSO entre métodos de importancia:')
print(f'  RF nativo ∩ TreeSHAP: {sorted(consensus)}')
print(f'  Features estables entre folds: {sorted(stable_top)}')

decision_log = {
    'fecha': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'sprint': 2,
    'mejor_configuracion_ablacion': best_config,
    'f1_por_config': {k: round(float(np.mean(v)), 4) for k, v in ablation_results.items()},
    'features_estables_entre_folds': stable_features,
    'consenso_rf_treeshap': sorted(consensus),
    'metodologia': {
        'ablaciones': 'Baseline vs Var1 vs Var2, F1 por cada fold (5-fold estratificado)',
        'importancia_arboreos': 'Random Forest nativo + TreeSHAP',
        'importancia_no_arboreos': 'Naive Bayes + KernelSHAP (muestra de 40 filas)',
        'estabilidad': 'top-5 features comparado entre 5 folds',
    },
    'decision': (
        f'Se adopta la configuración {best_config} por evidencia de ablación. '
        f'Las features estables ({len(stable_features)}) se priorizan por aparecer '
        f'consistentemente en el top-5 entre folds.'
    ),
}

with open('../logs/sprint2_decision_features.json', 'w', encoding='utf-8') as f:
    json.dump(decision_log, f, indent=2, ensure_ascii=False)

print('\n' + '='*60)
print('  DECISIÓN FINAL — SELECCIÓN DE FEATURES (basada en evidencia)')
print('='*60)
print(f'  1. Mejor config (ablación): {best_config}')
print(f'  2. F1 medio: {np.mean(ablation_results[best_config]):.4f}')
print(f'  3. Features estables: {len(stable_features)}')
print(f'  4. Consenso RF+TreeSHAP: {len(consensus)} features')
print('='*60)
print('✅ Log de decisión guardado: logs/sprint2_decision_features.json')

---
## ✅ Resumen del Sprint 2

| Punto del entregable | Cómo se cubrió |
|---|---|
| **Evidencia (no intuición)** | Toda decisión respaldada con F1, SHAP y estabilidad |
| **Ablaciones por fold** | Baseline/Var1/Var2 evaluadas en los 5 folds |
| **Importancia árboles → TreeSHAP** | Random Forest + TreeSHAP |
| **No-arbóreos → KernelSHAP** | Naive Bayes + KernelSHAP (muestra) |
| **Estabilidad top-5 entre folds** | Conteo de apariciones y umbral ≥4/5 |

### Archivos generados en `logs/`
- `sprint2_ablaciones.png` — comparación por fold
- `sprint2_importancia_rf.png` — importancia nativa RF
- `sprint2_treeshap.png` — TreeSHAP
- `sprint2_kernelshap.png` — KernelSHAP
- `sprint2_estabilidad_folds.png` — estabilidad top-5
- `sprint2_decision_features.json` — log de decisión
